In [1]:
import torch

In [2]:
# gradient를 추적할 입력 벡터를 생성한다.
x = torch.arange(
    4.0,
    requires_grad=True
)

# 원소별 제곱이므로 출력 y도 벡터다.
y = x * x

print("x:", x)
print("y:", y)
print("x shape:", x.shape)
print("y shape:", y.shape)

x: tensor([0., 1., 2., 3.], requires_grad=True)
y: tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>)
x shape: torch.Size([4])
y shape: torch.Size([4])


In [ ]:
# Non-스칼라 y에서 인자 없이 backward()를 호출하면 오류가 발생

try:
    y.backward()
except RuntimeError as error:
    print("Expected error:")
    print(error)
    
    
# 다음 실습을 위해 계산 그래프를 새로 생성
y = x * x

Expected error:
grad can be implicitly created only for scalar outputs


In [4]:
# 출력 y의 각 원소에서 upstream gradient 1이 들어온다고 지정
# 이는 L=sum(y)를 미분하는 것과 같음.
upstream_gradient = torch.ones_like(y)

y.backward(gradient=upstream_gradient)

expected_gradient = 2 * x.detach()

print("Upstream gradient:", upstream_gradient)
print("x.grad:", x.grad)
print("Expected gradient:", expected_gradient)

Upstream gradient: tensor([1., 1., 1., 1.])
x.grad: tensor([0., 2., 4., 6.])
Expected gradient: tensor([0., 2., 4., 6.])


In [5]:
# 기존 gradient를 초기화하고 새로운 계산 그래프
x.grad.zero_()
y = x * x

# 벡터 y를 sum으로 명시적으로 scalar로 축소함.
loss = y.sum()
loss.backward()

print("Gradient from y.sum():", x.grad)

assert torch.equal(
    x.grad,
    2 * x.detach(),
)

Gradient from y.sum(): tensor([0., 2., 4., 6.])


In [7]:
# 이번에는 출력 원소마다 서로 다른 upstream gradient를 전달
x.grad.zero_()
y = x * x

upstream_gradient = torch.tensor([
    1.0,
    2.0,
    3.0,
    4.0,
])

y.backward(gradient=upstream_gradient)

# yᵢ=xᵢ²이므로 각 gradient는 vᵢ·2xᵢ가 된다.
expected_weighted_gradient = (
    upstream_gradient
    * 2
    * x.detach()
)

print("Weighted upstream gradient:", upstream_gradient)
print("x.grad:", x.grad)
print("Expected gradient:", expected_weighted_gradient)

assert torch.equal(
    x.grad,
    torch.tensor([0.0, 4.0, 12.0, 24.0]),
)

Weighted upstream gradient: tensor([1., 2., 3., 4.])
x.grad: tensor([ 0.,  4., 12., 24.])
Expected gradient: tensor([ 0.,  4., 12., 24.])


In [8]:
# y=x²의 전체 Jacobian을 직접 구성해 backward 결과와 비교한다.
# 실제 역전파는 이 큰 행렬을 명시적으로 만들지 않는다.
jacobian = torch.diag(2 * x.detach())

vector_jacobian_product = (
    jacobian.T
    @ upstream_gradient
)

print("Jacobian:")
print(jacobian)

print("\nJ.T @ upstream_gradient:")
print(vector_jacobian_product)

Jacobian:
tensor([[0., 0., 0., 0.],
        [0., 2., 0., 0.],
        [0., 0., 4., 0.],
        [0., 0., 0., 6.]])

J.T @ upstream_gradient:
tensor([ 0.,  4., 12., 24.])
